### **Day 8: Window Functions**

Yesterday, we mastered aggregations and joins, noting that standard aggregations collapse your rows—turning many rows into a single summary row. Today, we are going to explore **Window Functions**.

Window functions are powerful because they allow you to calculate aggregates, rankings, or running totals across a specific group of rows, but **without collapsing the rows**. Every single input row retains its unique identity in the final output DataFrame. This is an essential tool for advanced analytics, time-series data, and financial reporting.

**Today's Objective**

By the end of this session, you will understand the concept of a "window" of data in a distributed system, how to partition and order data within a window, and how to apply ranking, analytic, and aggregate window functions efficiently.

**1. What is a Window Function?**

To understand a window function, contrast it with a traditional `groupBy`. If you group an employee dataset by `Department` to find the maximum salary, Spark collapses the dataset, returning only one row per department. You lose the names of the individuals who earned those salaries.

A **Window Function** performs a calculation over a specified group of rows (called the "window frame") that are relationally connected to the current row.

For example, for every employee row, a window function can look at their department, find the highest salary in that department, and append that maximum value as a *new column* right next to that employee's name and individual salary.

**2. The Three Components of a Window Specification**

To implement a window function, you must define a **Window Specification**. This tells PySpark exactly how to group, sort, and bound the rows for the calculation. A window spec consists of three structural parts:

*A. Partition By (The Grouping Boundary)*

This defines how the data is split into groups. It is completely independent of cluster partitions. For example, if you partition by `Department`, Spark ensures that the calculations for the "Sales" department do not bleed into the "Engineering" department.

*B. Order By (The Sequence)*

This defines how the rows are sorted *inside* each partition. Sorting is mandatory for tracking sequences, such as identifying who has the highest or lowest sales, or calculating a running total over time.

*C. Rows Between / Range Between (The Frame Boundaries)*

This defines exactly which rows relative to the current row should be included in the calculation.

* *Example:* You can define a frame that looks at the `current row` and the `2 rows preceding it` to calculate a 3-day moving average.
* If you do not specify a frame boundary, Spark defaults to including everything from the start of the partition up to the current row.


**3. The Three Categories of Window Functions**

Once your window frame is defined, you can apply three types of mathematical operations across it:

*Category 1: Ranking Functions*

These functions assign a sequential number or rank to rows within the sorted window.

* **`row_number()`**: Assigns a unique, sequential integer starting at 1. If two rows have identical values, it arbitrarily assigns them different numbers.
* **`rank()`**: Assigns a rank based on value. If two rows tie for 1st place, they both get a rank of 1. The next row gets a rank of 3 (it skips a rank).
* **`dense_rank()`**: Identical to `rank()`, but it **never skips a number**. If two rows tie for 1st place, they both get a rank of 1, and the next row gets a rank of 2.

*Category 2: Analytic / Value Functions*

These functions allow you to peek forward or backward into other rows within the window without performing complex joins.

* **`lag(column, n)`**: Fetches the value of a column from `n` rows *before* the current row. This is perfect for calculating day-over-day growth or matching a current transaction with the previous transaction.
* **`lead(column, n)`**: Fetches the value of a column from `n` rows *after* the current row.

*Category 3: Aggregate Functions*

You can use standard mathematical aggregations inside a window spec to create running metrics.

* **`sum()`**, **`avg()`**, **`min()`**, **`max()`**: When combined with an `orderBy` clause, these functions automatically calculate running totals or running averages row by row.

**4. The Cluster Performance Cost of Windows**

While window functions are incredibly useful, they come with a high architectural cost that a PySpark expert must manage:

> **The Single-Machine Bottleneck:** To calculate a window function, Spark must guarantee that all rows belonging to the same `PartitionBy` group are collected on the **exact same physical Executor machine**.

If you partition by a column that has very few unique values (for example, partitioning a billion-row dataset by `Gender` which only has a few distinct keys), Spark will be forced to shuffle hundreds of millions of rows over the network onto just two or three worker machines. This creates massive data skew, causes memory overloads, and can completely stall your cluster.